<a href="https://colab.research.google.com/github/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/ProyectoAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mini-Proyecto Aprendizaje Profundo
### Autoras: Ana Blasco & Adriana Marí

# Asistente Multimodal de Salud para Personas Mayores

## 1. Descripción del proyecto

El objetivo de este proyecto es desarrollar un asistente inteligente que ayude a personas mayores a gestionar aspectos básicos de su salud y bienestar diario de forma sencilla. Muchas personas mayores tienen dificultades para recordar correctamente indicaciones médicas, como la posología de sus medicaciones o para leer documentos con letra pequeña, como los prospectos. Este sistema pretende facilitar dichas tares mediante el uso de inteligencia artificial y diferentes modalidades de entrada y salida de información.

El asistente permitirá interactuar con el usuario principalmente a través de la voz, aunque también podrá procesar otros tipos de entrada, como imágenes o documentos. Por ejemplo, una persona podrá pedir por voz qué medicamentos debe tomar ese día, o hacer una foto de una prescripción médica para que el sistema la analice y le lea las instrucciones de forma clara y comprensible. De esta manera, el sistema recibe información médica potencialmente compleja y la adaptada a voluntad del usuario.

El sistema se basa en un *pipeline* de modelos de inteligencia artificial preentrenados que permiten procesar diferentes tipos de datos. En primer lugar, si la entrada es audio, se utiliza un modelo de speech-to-text (S2T) para convertir la voz del usuario en texto y que un modelo de lenguaje (text-to-text, T2T) la analice. Si la entrada es una imagen o documento, se utilizan técnicas de lenguaje visual (VL) para extraer la información relevante en forma de texto.

Finalmente, el sistema genera una respuesta que puede presentarse de diferentes maneras. Por un lado, todas las respuestas se proporcionan en forma de voz utilizando un modelo de text-to-speech (T2S) para facilitar la comprensión y algunas también proporcionan una respuesta parcial escrita, acorde a la hablada. Por otro lado, también se pueden generar salidas visuales, como es el resumen de medicación del día, que es un recordatorio visual de los medicamentos que el usuario debe tomar ese día con sus respectivas dosis.

## 2. Diagrama de Flujo

El astistente es multimodal porque, com habíamos dicho anteriormente, acepta diferentes tipos de entrada: voz, imágenes o documentos PDF. Dependiendo del tipo de entrada, se utilizan distintos modelos de inteligencia artificial para convertir la información a texto. Después, un modelo de lenguaje interpreta la petición del usuario y genera una respuesta que puede presentarse en diferentes formatos, como audio, texto o tabla HTML para facilitar la comprensión.

<img src="imagenes/estructura.png" alt="Diagrama de flujo">

## 3. Fases del desarrollo del asistente

Decidimos separar el desarrollo del asistente multimodal en dos etapas diferenciadas: la fase A y la fase B. Por un lado, el objetivo principal que motiva la fase A es asentar la lógica de la interacción asitente-usuario, decidiendo como mostrar y acceder a las opciones que el asistente ofrece. Por otro lado, la fase B se focaliza en las funciones concretas que llevarán a cabo la ya mencionada interacción; aquellas funciones que recabarán la información e implementarán los modelos preentrenados. Aunque en este documento se expliquen ambas fases, remitimos al lector a los archivos `FaseA.ipynb` y `FaseB.ipynb` para más información y detalles.

### 3.1. Fase A: gestión del asistente

Lo primero que fijamos en esta fase fue la forma en la que se guardaría la información del usuario: la _memoria_ del asistente es un archivo en formato _JSON_ que contiene un diccionario con tres campos: el nombre por el que dirigirse al usuario, el historial de medicaciones aportado y una lista indicando cuáles han sido las últimas medicinas incorporadas al historial.

Después de esto creamos una función de presentación para que, si existe el archivo de memoria se salude por su nombre al usuario y se pase directamente al menú principal. En caso contrario el asistente se presenta y le pide al usuario que le indique oralmente un nombre por que poder referirse a él/ella. Cabe recalcar que, a excepción de las imágenes o documentos PDF, decidimos que todas las entradas por parte del usuario fueran en audio, de forma que salvamos dos obstáculos: las dificultades que muchas veces presentan las personas mayores para leer y/o escribir en un teclado, y la accesibilidad del asistente, ya que de esta forma la interacción resulta en un diálogo. Para posibilitar dicho diálogo, elegimos el modelo Whisper AI para pasar el audio del usuario a texto, el modelo QWen2.5-3B-instruct para procesar el texto y el modelo Edge TTS para brindarle voz al asistente. Como podemos ver, a pesar de que en esta fase todavía no entramos de lleno en el uso de modelos multimodales con el objetivo que perseguimos, sí que son una pieza clave.

Otro punto que abordamos en esta fase fue la creación dos menús: el principal y el de ajustes. Desde el menú principal, a través del cual se puede seleccionar los distintos recursos desarrollados en la fase B y/o acabar la sesión, también se puede acceder al humilde menú de opciones, donde el usuario podrá elegir entre cambiar el nombre, borrar todo su historial de medicinas o volver al menú principal.

Por último, definimos la función global que unía todos los elementos indicados hasta ahora, siguiendo el esquema general que muestra la Imagen 1.

![Imagen 1. Estructura de la gestión del asistente](https://github.com/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/'imagenes/estructura.png'?raw=1)


### 3.2. Fase B: Procesamiento Multimodal de la Información

Esta fase, la más extensa, se centra en definir las fuciones necesarias para abordar las tareas que tiene como objetivo el asistente. Recordemos que dichas tareas eran:

- Registrar un medicamento nuevo en el historial del a través de imágenes.
- Modificar o eliminar un medicamento del historial a través de la voz.
- Resumir la medicación de ese día mediante voz y una tabla HTML.
- Responder preguntas sencillas del usuario a través de la voz.
- Leer documentos PDF o imágenes en voz alta.

Por tanto, decidimos crear una función para cada opción, con las cuales integraremos cada funcionalidad en el menú principal de la fase A. A continuación explicamos detalladamente cada una.


#### 3.2.1. Registrar un medicamento nuevo en el historial del a través de imágenes

Para registrar un medicamento nuevo en el historial del usuario a través de imágenes, decidimos crear la función `ejecutar_opcion_1(memoria)` que tiene como argumento de entrada el historial del usuario. Pero antes de explicarla, hablemos de las funciones que usa y de las que no hemos hablado en la fase A: `analizar_receta()`, `mostrar_recordatorios` y `registrar_en_memoria`. 

La función `analizar_receta(ruta_imagen, memoria)` es la encargada de procesar la imagen que el usuario le proporciona, que se supone que es una receta médica. Para ello, se basa en un modelo de VL QWen2.5-3B-instruct, que es capaz de interpretar imágenes y extraer información relevante. En este caso, el modelo analiza la receta y extrae los nombres de los medicamentos, las dosis y las fechas de finalización de tratamientos. La función devuelve esta información en formato _JSON_.

La función `mostrar_recordatorios(nuevos_datos)` es la encargada de generar una texto breve resumiendo la nueva medicación añadida a la memoria del asistente. Primero recibe como argumento la salida de la función `analizar_receta` y la compara "manualmente" con los medicamentos registrados en memoria. Mediante esta comparación discrimina qué medicamentos han sido las últimas adiciones al historial y los introduce en el campo de memoria `ultimas_adiciones`  Luego, genera una breve texto que resume las últimas medicaciones añadidas, que es el que se le muestra al usuario para que lo confirme.